# 🌍 Interactive World GDP Map
### Toggle between: GDP (Actuals) · GDP Growth Forecast (IMF) · GDP Per Capita
**Data Sources:**
- **GDP (Actuals):** World Bank API — `NY.GDP.MKTP.CD`
- **GDP Per Capita:** World Bank API — `NY.GDP.PCAP.CD`
- **GDP Growth Forecast:** IMF World Economic Outlook API (latest projections)

In [1]:
# ─── Cell 1 · Install dependencies if needed ───────────────────────────────
# Uncomment if running for the first time
# !pip install requests pandas plotly numpy

In [2]:
# ─── Cell 2 · Imports ──────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports done')

✅ Imports done


In [3]:
# ─── Cell 3 · Fetch World Bank GDP (Actuals) ───────────────────────────────

def fetch_worldbank(indicator, label):
    """Fetch latest available value per country from the World Bank API."""
    url = (
        f"https://api.worldbank.org/v2/country/all/"
        f"indicator/{indicator}"
        f"?format=json&per_page=30000"
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.json()

    records = []
    for row in raw[1]:
        if row["value"] is not None and row["countryiso3code"]:
            records.append({
                "COUNTRY" : row["country"]["value"],
                "CODE"    : row["countryiso3code"],
                "YEAR"    : row["date"],
                label     : row["value"]
            })

    df = pd.DataFrame(records)
    df_latest = (
        df.sort_values("YEAR", ascending=False)
          .drop_duplicates(subset=["CODE"])
          .reset_index(drop=True)
    )
    print(f'  ✅ {label}: {len(df_latest)} countries  |  latest year in data: {df_latest["YEAR"].max()}')
    return df_latest


print('⏳ Fetching World Bank GDP (Actuals)...')
df_gdp = fetch_worldbank("NY.GDP.MKTP.CD", "GDP")

print('⏳ Fetching World Bank GDP Per Capita...')
df_pcap = fetch_worldbank("NY.GDP.PCAP.CD", "GDP_PER_CAPITA")

⏳ Fetching World Bank GDP (Actuals)...
  ✅ GDP: 258 countries  |  latest year in data: 2024
⏳ Fetching World Bank GDP Per Capita...
  ✅ GDP_PER_CAPITA: 258 countries  |  latest year in data: 2024


In [4]:
# ─── Cell 4 · Fetch IMF WEO GDP Growth Forecast ────────────────────────────
#
# IMF WEO SDMX API — indicator NGDP_RPCH = Real GDP growth rate (% change)
# We pull the latest available forecast year (typically current + 1 year)

def fetch_imf_gdp_growth():
    """
    Fetch IMF World Economic Outlook GDP growth forecasts.
    Returns a DataFrame with CODE, COUNTRY, GDP_GROWTH, FORECAST_YEAR.
    """
    # IMF Compact JSON API
    url = (
        "https://www.imf.org/external/datamapper/api/v1/NGDP_RPCH"
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.json()

    # raw['values']['NGDP_RPCH'] → { 'AFG': {'2020': 2.1, ...}, ... }
    country_data = raw.get('values', {}).get('NGDP_RPCH', {})

    # Also grab labels (country names) if available
    labels = raw.get('entities', {}).get('countries', {})

    records = []
    for code, years in country_data.items():
        if not years:
            continue
        # Pick the most recent year that has a value
        latest_year = max(years.keys())
        value = years[latest_year]
        if value is None:
            continue
        country_name = labels.get(code, {}).get('label', code)
        records.append({
            "CODE"         : code,
            "COUNTRY"      : country_name,
            "GDP_GROWTH"   : float(value),
            "FORECAST_YEAR": latest_year
        })

    df = pd.DataFrame(records)
    print(f'  ✅ IMF GDP Growth: {len(df)} entries  |  latest forecast year: {df["FORECAST_YEAR"].max()}')
    return df


print('⏳ Fetching IMF GDP Growth Forecasts...')
df_growth = fetch_imf_gdp_growth()
df_growth.head()

⏳ Fetching IMF GDP Growth Forecasts...
  ✅ IMF GDP Growth: 229 entries  |  latest forecast year: 2031


,CODE,COUNTRY,GDP_GROWTH,FORECAST_YEAR
0,SDN,SDN,4.5,2031
1,AFG,AFG,3.0,2025
2,ALB,ALB,3.2,2031
3,DZA,DZA,2.6,2031
4,AND,AND,1.5,2031


In [5]:
# ─── Cell 5 · Merge all three datasets ─────────────────────────────────────

# Start with GDP as the base (has the most complete country list)
df_merged = df_gdp[["COUNTRY", "CODE", "YEAR", "GDP"]].copy()
df_merged.rename(columns={"YEAR": "GDP_YEAR"}, inplace=True)

# Merge per-capita
df_merged = df_merged.merge(
    df_pcap[["CODE", "GDP_PER_CAPITA", "YEAR"]].rename(columns={"YEAR": "PCAP_YEAR"}),
    on="CODE", how="left"
)

# Merge IMF growth
df_merged = df_merged.merge(
    df_growth[["CODE", "GDP_GROWTH", "FORECAST_YEAR"]],
    on="CODE", how="left"
)

# Log-scale GDP for better choropleth contrast
df_merged["GDP_LOG"] = np.log10(df_merged["GDP"].clip(lower=1))
df_merged["PCAP_LOG"] = np.log10(df_merged["GDP_PER_CAPITA"].clip(lower=1))

# Friendly labels for hover
def fmt_usd(val, billions=True):
    if pd.isna(val):
        return "N/A"
    if billions:
        b = val / 1e9
        if b >= 1000:
            return f"${b/1000:,.2f}T"
        return f"${b:,.1f}B"
    return f"${val:,.0f}"

df_merged["GDP_LABEL"]   = df_merged["GDP"].apply(lambda v: fmt_usd(v, True))
df_merged["PCAP_LABEL"]  = df_merged["GDP_PER_CAPITA"].apply(lambda v: fmt_usd(v, False))
df_merged["GROWTH_LABEL"]= df_merged["GDP_GROWTH"].apply(
    lambda v: f"{v:+.2f}%" if not pd.isna(v) else "N/A"
)

print(f'✅ Merged dataset: {len(df_merged)} countries')
print(df_merged[["COUNTRY","CODE","GDP_LABEL","PCAP_LABEL","GROWTH_LABEL","FORECAST_YEAR"]].head(10).to_string(index=False))

✅ Merged dataset: 258 countries
                       COUNTRY CODE GDP_LABEL PCAP_LABEL GROWTH_LABEL FORECAST_YEAR
   Africa Eastern and Southern  AFE    $1.24T     $1,615          N/A           NaN
                        Belize  BLZ     $3.2B     $7,681       +2.00%          2031
              Puerto Rico (US)  PRI   $126.0B    $39,344       +0.80%          2031
                      Barbados  BRB     $7.5B    $26,545       +2.00%          2031
                      Portugal  PRT   $313.3B    $29,292       +1.60%          2031
                       Belarus  BLR    $76.0B     $8,318       +0.90%          2031
                       Belgium  BEL   $671.4B    $56,615       +1.30%          2031
                        Poland  POL   $917.8B    $25,104       +2.50%          2031
Central Europe and the Baltics  CEB    $2.46T    $24,605          N/A           NaN
                   Philippines  PHL   $461.6B     $3,985       +6.00%          2031


In [ ]:
# ─── Cell 6 · Build interactive toggle map ─────────────────────────────────
#
# Three choropleth traces, one per metric, with Plotly updatemenus
# for a clean button-toggle UX.
#
# IMPORTANT FIX: when a button's `args` includes a layout-update dict,
# Plotly only patches the keys you explicitly list — it does NOT merge
# nested dicts automatically for `geo`. If `geo` styling isn't re-stated
# in every button (or template isn't pinned), some renderers reset it to
# the white default on click. We fix this by:
#   1) pinning template="plotly_dark" globally (a real safety net)
#   2) re-asserting full geo + paper/plot background in EVERY button
#   3) re-asserting the matching annotation (caption) text in EVERY button

import plotly.graph_objects as go

PROJ = "natural earth"

DARK_BG = "#0a0a1a"
LAND_BG = "#1a1a2e"

GEO_STYLE = dict(
    projection_type = PROJ,
    showcoastlines  = True,  coastlinecolor="rgba(255,255,255,0.4)",
    showland        = True,  landcolor=LAND_BG,
    showocean       = True,  oceancolor=DARK_BG,
    showlakes       = True,  lakecolor=DARK_BG,
    showframe       = False,
    bgcolor         = DARK_BG,
)

# ── colour scales ──────────────────────────────────────────────────────────
# Each metric gets a scale chosen for what the number actually means:
#  - GDP            → sequential (low→high, one direction) — size of economy
#  - GDP per capita → sequential, DIFFERENT hue from GDP so the two are never
#                     visually confused even though both use a log scale
#  - GDP growth     → diverging around a meaningful zero — shrinking vs growing
SCALE_GDP    = "YlOrRd"          # yellow → red: total economic size (no negative direction)
SCALE_PCAP   = "Blues"           # light → dark blue: wealth per person, distinct hue from GDP
SCALE_GROWTH = "RdYlGn"          # red-yellow-green (growth = good is green)

# ── helper: build one choropleth trace ─────────────────────────────────────
def make_trace(df, z_col, text_col, name, colorscale, zmid=None,
               colorbar_title="", ticksuffix="", tickprefix=""):
    kw = dict(zmid=zmid) if zmid is not None else {}
    return go.Choropleth(
        locations       = df["CODE"],
        z               = df[z_col],
        text            = df["COUNTRY"],
        customdata      = df[[text_col, "GDP_YEAR", "PCAP_YEAR", "FORECAST_YEAR"]].values,
        hovertemplate   = (
            "<b>%{text}</b><br>"
            + name + ": <b>%{customdata[0]}</b><br>"
            "<extra></extra>"
        ),
        colorscale      = colorscale,
        autocolorscale  = False,
        reversescale    = False,
        marker_line_color="white",
        marker_line_width=0.4,
        colorbar=dict(
            title       = dict(text=colorbar_title, side="right"),
            thickness   = 14,
            len         = 0.75,
            tickprefix  = tickprefix,
            ticksuffix  = ticksuffix,
        ),
        **kw
    )


# ── Trace 1 · GDP (log scale for better contrast) ──────────────────────────
trace_gdp = make_trace(
    df_merged, "GDP_LOG", "GDP_LABEL",
    name="GDP",
    colorscale=SCALE_GDP,
    colorbar_title="GDP (log₁₀ USD)",
)

# ── Trace 2 · GDP Growth Forecast ──────────────────────────────────────────
growth_max = df_merged["GDP_GROWTH"].abs().quantile(0.97)   # clip extreme outliers
trace_growth = make_trace(
    df_merged, "GDP_GROWTH", "GROWTH_LABEL",
    name="IMF GDP Growth Forecast",
    colorscale=SCALE_GROWTH,
    zmid=0,
    colorbar_title="Growth (%)",
    ticksuffix="%",
)
# clamp colour range symmetrically around 0
trace_growth.zmin = -growth_max
trace_growth.zmax =  growth_max

# ── Trace 3 · GDP Per Capita (log) ─────────────────────────────────────────
trace_pcap = make_trace(
    df_merged, "PCAP_LOG", "PCAP_LABEL",
    name="GDP Per Capita",
    colorscale=SCALE_PCAP,
    colorbar_title="GDP/capita (log₁₀ USD)",
)


# ── Figure ──────────────────────────────────────────────────────────────────
fig = go.Figure(data=[trace_gdp, trace_growth, trace_pcap])

# Start with only trace_gdp visible
fig.data[0].visible = True
fig.data[1].visible = False
fig.data[2].visible = False


# ── Years for titles/captions ────────────────────────────────────────────
latest_gdp_yr    = df_merged['GDP_YEAR'].mode()[0]
latest_pcap_yr   = df_merged['PCAP_YEAR'].dropna().mode()[0] if df_merged['PCAP_YEAR'].notna().any() else ''
latest_growth_yr = df_merged['FORECAST_YEAR'].dropna().mode()[0] if df_merged['FORECAST_YEAR'].notna().any() else ''


# ── How-to-read captions (rendered BELOW the map, swapped per toggle) ──────
CAPTION_GDP = (
    "<b>How to read this map — GDP (Total Economic Output)</b><br>"
    "Colour shows each country's total GDP in USD, on a <b>log scale</b> "
    "(log scale is used because a handful of economies — the US, China — are "
    "thousands of times larger than most countries; without it, nearly every "
    "country would look the same dark colour). "
    "<b>Darker red = larger economy.</b> This measures total output, NOT how "
    "wealthy individual citizens are — a large population can produce a high "
    "GDP even with low income per person. Pair with GDP Per Capita for that view."
)

CAPTION_GROWTH = (
    "<b>How to read this map — GDP Growth Forecast (IMF)</b><br>"
    "Colour shows the projected <b>year-over-year % change</b> in real GDP, "
    "centred on zero. <b>Green = economy expected to grow, red = economy "
    "expected to shrink</b>, yellow ≈ flat. Unlike the other two maps this is "
    "a <b>forward-looking forecast</b>, not a measured actual — and it says "
    "nothing about the size of the economy, only its direction. A small "
    "country can show deep green while a giant economy shows pale yellow."
)

CAPTION_PCAP = (
    "<b>How to read this map — GDP Per Capita</b><br>"
    "Colour shows GDP divided by population — average economic output per "
    "person, in USD, on a <b>log scale</b> for the same reason as the GDP map. "
    "<b>Darker blue = higher income per person.</b> This is the better proxy "
    "for average living standards. Small, wealthy nations can outrank giant "
    "economies here even though they show pale on the GDP map."
)


def caption_annotation(text):
    """Build the below-map caption as a layout annotation (paper-anchored).

    FIX: Plotly annotations do NOT auto-wrap text to a pixel `width` --
    `width` only sets the background box size; overflowing text gets
    clipped by the plotting area instead of wrapping (this caused the
    truncated caption). We manually wrap into <br>-separated lines using
    textwrap BEFORE handing the string to Plotly.
    """
    import textwrap
    if "<br>" in text:
        heading, body = text.split("<br>", 1)
    else:
        heading, body = "", text

    wrapped_body = "<br>".join(textwrap.wrap(body.strip(), width=118))
    full_text = heading + ("<br>" + wrapped_body if wrapped_body else "")

    return dict(
        text          = full_text,
        showarrow     = False,
        xref          = "paper", yref="paper",
        x             = 0.5,     y  = -0.14,
        xanchor       = "center", yanchor="top",
        align         = "center",
        font          = dict(size=12.5, color="#d8d8e8"),
        bgcolor       = "#15152b",
        bordercolor   = "#3a3a5e",
        borderwidth   = 1,
        borderpad     = 12,
        width         = 1000,
    )


TITLE_GDP = (
    f"<b>World GDP — Latest Actuals ({latest_gdp_yr})</b>  |  World Bank  |  Log scale"
)
TITLE_GROWTH = (
    f"<b>GDP Growth Rate Forecast ({latest_growth_yr})</b>  |  IMF World Economic Outlook"
)
TITLE_PCAP = (
    f"<b>GDP Per Capita — Latest Actuals ({latest_pcap_yr})</b>  |  World Bank  |  Log scale"
)


# ── Toggle buttons ───────────────────────────────────────────────────────
def button_layout(visible, title, caption_text):
    return [
        {"visible": visible},
        {
            "title": title,
            "geo": GEO_STYLE,
            "paper_bgcolor": DARK_BG,
            "plot_bgcolor": DARK_BG,
            "font": dict(color="white"),
            "annotations": [caption_annotation(caption_text)],
        },
    ]


buttons = [
    dict(
        label  = f"\U0001F4CA  GDP  ({latest_gdp_yr})",
        method = "update",
        args   = button_layout([True, False, False], TITLE_GDP, CAPTION_GDP),
    ),
    dict(
        label  = f"\U0001F4C8  GDP Growth Forecast  ({latest_growth_yr})",
        method = "update",
        args   = button_layout([False, True, False], TITLE_GROWTH, CAPTION_GROWTH),
    ),
    dict(
        label  = f"\U0001F4B0  GDP Per Capita  ({latest_pcap_yr})",
        method = "update",
        args   = button_layout([False, False, True], TITLE_PCAP, CAPTION_PCAP),
    ),
]
fig.update_layout(
    template = "plotly_dark",   # safety net: dark is now the BASE theme, not an override
    title=dict(
        text  = TITLE_GDP,
        x     = 0.5,
        xanchor="center",
        font  = dict(size=16, color="white")
    ),
    geo = GEO_STYLE,
    annotations = [caption_annotation(CAPTION_GDP)],
    updatemenus=[
        dict(
            type       = "buttons",
            direction  = "right",
            x          = 0.5,
            xanchor    = "center",
            y          = 1.08,
            yanchor    = "top",
            buttons    = buttons,
            pad        = dict(r=10, t=6, b=6, l=10),
            showactive = True,
            bgcolor    = "#3d3d6b",
            bordercolor= "#7a7ab0",
            borderwidth= 1.5,
            font       = dict(color="#0a0a1a", size=12.5),
        )
    ],
    paper_bgcolor = DARK_BG,
    plot_bgcolor  = DARK_BG,
    font          = dict(color="white"),
    margin        = dict(l=0, r=0, t=110, b=200),
    
    width         = 1400,
    height        = 800,
    hoverlabel    = dict(
        bgcolor   = "#1e1e3f",
        bordercolor="#5a5aaa",
        font      = dict(size=13, color="white")
    ),
)

fig.show()
#print('✅ Map rendered — use the buttons above the map to toggle metrics; the caption below the map explains how to read each one')

✅ Map rendered — use the buttons above the map to toggle metrics; the caption below the map explains how to read each one


In [11]:
# ─── Cell 7 · (Optional) Quick data summary tables ─────────────────────────

print('=== Top 15 Economies by GDP ===')
top_gdp = (
    df_merged.sort_values("GDP", ascending=False)
    [["COUNTRY", "GDP_LABEL", "GDP_YEAR"]]
    .dropna()
    .head(15)
    .reset_index(drop=True)
)
top_gdp.index += 1
print(top_gdp.to_string())

print('\n=== Top 15 by GDP Per Capita ===')
top_pcap = (
    df_merged.sort_values("GDP_PER_CAPITA", ascending=False)
    [["COUNTRY", "PCAP_LABEL", "PCAP_YEAR"]]
    .dropna()
    .head(15)
    .reset_index(drop=True)
)
top_pcap.index += 1
print(top_pcap.to_string())

print('\n=== Fastest Growing Economies (IMF Forecast) ===')
top_growth = (
    df_merged.sort_values("GDP_GROWTH", ascending=False)
    [["COUNTRY", "GROWTH_LABEL", "FORECAST_YEAR"]]
    .dropna()
    .head(15)
    .reset_index(drop=True)
)
top_growth.index += 1
print(top_growth.to_string())

=== Top 15 Economies by GDP ===
                                        COUNTRY GDP_LABEL GDP_YEAR
1                                         World  $110.98T     2024
2                                  OECD members   $67.50T     2024
3                     Post-demographic dividend   $62.79T     2024
4                              IDA & IBRD total   $43.82T     2024
5                                     IBRD only   $40.93T     2024
6                           Low & middle income   $39.40T     2024
7                                 Middle income   $38.93T     2024
8                           East Asia & Pacific   $32.02T     2024
9                                 North America   $31.00T     2024
10                    Late-demographic dividend   $29.98T     2024
11                        Europe & Central Asia   $29.29T     2024
12                                United States   $28.75T     2024
13  East Asia & Pacific (excluding high income)   $22.26T     2024
14   East Asia & Pacific (IDA 